In [ ]:
import torch
import pandas as pd
import sklearn
import altair as alt
import numpy as np
import yaml
from omegaconf import OmegaConf
from ts2.train.main_cell_inference import SingleCellTempInferenceDataset
from tqdm import tqdm
import gzip
import pydicom
from PIL import Image
import gc
import matplotlib.pyplot as plt
from glob import glob

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import cv2


In [ ]:
from collections import defaultdict

def merge_list_of_dicts(dict_list):
    merged = defaultdict(list)
    for d in dict_list:
        for k, v in d.items():
            merged[k].extend(v)
    return dict(merged)

In [ ]:
@torch.no_grad()
def get_knn_logits(cf, train_embs, val_embs, train_labels):
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    torch.cuda.empty_cache()
    
    batch_size = cf["testing"]["knn"]["knn_params"]["batch_size"]
    all_scores = []
    nc = len(set(train_labels.tolist()))
    temb = train_embs.T.to(device)
    tl = train_labels.to(device)
    for k in tqdm(range(val_embs.shape[0] // batch_size + 1)):
        # find current minibatch
        start_coeff = batch_size * k
        end_coeff = min(batch_size * (k + 1),
                        val_embs.shape[0])  # leftover
        val_embs_k = val_embs[start_coeff:end_coeff].to(
            device)  # 1536 x 2048

        # knn predict on the minibatch
        _, pred_scores = knn_predict(
            val_embs_k,
            temb,
            tl,
            nc,
            knn_k=cf["testing"]["knn"]["knn_params"]["k"],
            knn_t=cf["testing"]["knn"]["knn_params"]["t"])

        # add to list
        all_scores.append(
            torch.nn.functional.normalize(pred_scores, p=1, dim=1).cpu())
        torch.cuda.empty_cache()

    return torch.vstack(all_scores)


# code for kNN prediction from here:
# https://colab.research.google.com/github/facebookresearch/moco/blob/colab-notebook/colab/moco_cifar10_demo.ipynb
def knn_predict(feature, feature_bank, feature_labels, classes: int,
                knn_k: int, knn_t: float):
    """Helper method to run kNN predictions on features based on a feature bank
    Args:
        feature: Tensor of shape [B, D] consisting of N D-dimensional features
        feature_bank: Tensor of shape [N, D], a database of features used for kNN
        feature_labels: Labels for the features in our feature_bank
        classes: Number of classes (e.g. 10 for CIFAR-10)
        knn_k: Number of k neighbors used for kNN
        knn_t: Temperature in kNN, low temperature leads to more weighted kNN.
    """
    # compute cos similarity between each feature vector and feature bank ---> [B, N]
    sim_matrix = torch.mm(feature, feature_bank)
    # [B, K]
    sim_weight, sim_indices = sim_matrix.topk(k=knn_k, dim=-1)
    # [B, K]
    sim_labels = torch.gather(feature_labels.expand(feature.shape[0], -1),
                              dim=-1,
                              index=sim_indices)
    # we do a reweighting of the similarities
    sim_weight = (sim_weight / knn_t).exp()
    # counts for each class
    one_hot_label = torch.zeros(feature.shape[0] * knn_k,
                                classes,
                                device=sim_labels.device)
    # [B*K, C]
    one_hot_label = one_hot_label.scatter(dim=-1,
                                          index=sim_labels.view(-1, 1),
                                          value=1.0)
    
    # weighted score ---> [B, C]
    pred_scores = torch.sum(one_hot_label.view(feature.size(0), -1, classes) *
                            sim_weight.unsqueeze(dim=-1),
                            dim=1)
    pred_labels = pred_scores.argsort(dim=-1, descending=True)
    return pred_labels, pred_scores

In [ ]:
def gmm_fit(X, **kwargs):
    gmm = GaussianMixture(**kwargs)
    gmm.fit(X)
    return gmm

def gmm_inference(gmm: GaussianMixture, mixture_score: np.ndarray, inf_embs: np.ndarray, 
                  mixture_hard: bool = True, point_hard: bool = False) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute the predicted positive score for each input embedding based on GMM responsibilities
    and per-component positive likelihoods.

    Args:
        gmm: A fitted sklearn GaussianMixture model.
        mixture_score: Array of shape (n_components,), where each entry is the likelihood of a component being positive.
        inf_embs: Array of shape (n_samples, n_features), new data points for inference.
        mixture_hard: If True, binarize the mixture scores using a 0.5 threshold.
        point_hard: If True, perform hard assignment of points to clusters (one-hot responsibility).

    Returns:
        pos_scores: Array of shape (n_samples,) with the predicted positive scores.
        responsibilities: Array of shape (n_samples, n_components) with the (soft or hard) responsibilities.
    """
    mixture_score = np.asarray(mixture_score)
    assert mixture_score.shape[0] == gmm.n_components, "Mismatch between score vector and number of GMM components."

    if mixture_hard:
        mixture_score = (mixture_score > 0.5).astype(float)

    # Posterior responsibilities: shape (n_samples, n_components)
    responsibilities = gmm.predict_proba(inf_embs)

    if point_hard:
        # Convert to one-hot hard assignment
        hard_assignments = np.argmax(responsibilities, axis=1)
        responsibilities = np.eye(gmm.n_components)[hard_assignments]

    # Weighted average of per-component scores
    pos_scores = responsibilities @ mixture_score

    return pos_scores, responsibilities

In [ ]:
# ours:
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/6778e5d1_May27-15-59-58_sd1000_dev_tune0/models/eval/training_124999/941f49cb_May29-01-43-17_sd1000_smpt_BENCHDB_SE_epoch0-step124999_tune0/predictions/train_predictions.pt.gz") as fd:
#    db_data = torch.load(fd)

# dv2
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/89d3ad98_May23-13-58-49_sd1000_dev_tune0/models/eval/training_124999/0e97d769_May26-11-14-43_sd1000_smpt_BENCHDB_SE_epoch0-step124999_tune3/predictions/train_predictions.pt.gz") as fd:
#    db_data = torch.load(fd)

#db_data = pd.DataFrame({
#    "path":db_data["path"],
#    "label":db_data["label"],
#    "embeddings": [i for i in db_data["embeddings"]]
#})


# srh7
#db_data = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/6778e5d1_May27-15-59-58_sd1000_dev_tune0/models/eval/training_124999/2510a2cb_May30-19-10-14_sd1000_INFDB_dev/predictions/pred.pt")
#db_data = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/89d3ad98_May23-13-58-49_sd1000_dev_tune0/models/eval/training_124999/17b1bb09_May30-19-44-14_sd1000_INFDB_dev/predictions/pred.pt")

# only gliomas
db_data = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/6778e5d1_May27-15-59-58_sd1000_dev_tune0/models/eval/training_124999/e2e43218_Jun03-18-51-29_sd1000_INFDB_dev/predictions/pred.pt")
# /nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/89d3ad98_May23-13-58-49_sd1000_dev_tune0/models/eval/training_124999/6b91de8d_Jun03-19-13-20_sd1000_INFDB_dev

# g2m, no linear norm
#db_data = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/6778e5d1_May27-15-59-58_sd1000_dev_tune0/models/eval/training_124999/dd7f97e0_Jun06-21-19-30_sd1000_INFDB_NOIN_dev/predictions/pred.pt")


db_data = pd.DataFrame(db_data)
db_embs = torch.stack(db_data["embeddings"].tolist())

In [ ]:
# ours
#data = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/6778e5d1_May27-15-59-58_sd1000_dev_tune0/models/eval/training_124999/14627c24_May29-14-12-07_sd1000_INF_dev/predictions/pred.pt")

# ours mouse
data = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/6778e5d1_May27-15-59-58_sd1000_dev_tune0/models/eval/training_124999/4100a9ab_Jun01-02-43-45_sd1000_INF_dev/predictions/pred.pt")

# ours ucsf
#data = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/6778e5d1_May27-15-59-58_sd1000_dev_tune0/models/eval/training_124999/5678c19d_Jun01-19-43-34_sd1000_INF_dev/predictions/pred.pt")

# ours ucsf no linear norm
#/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/6778e5d1_May27-15-59-58_sd1000_dev_tune0/models/eval/training_124999/313e77cc_Jun06-22-42-25_sd1000_INF_dev

# ======

# dv2
#data = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/89d3ad98_May23-13-58-49_sd1000_dev_tune0/models/eval/training_124999/f6cafa56_May29-18-37-15_sd1000_INF_dev/predictions/pred.pt")

# dv2 mouse
# /nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/89d3ad98_May23-13-58-49_sd1000_dev_tune0/models/eval/training_124999/a668638d_Jun01-03-14-14_sd1000_INF_dev/predictions/pred.pt")

# dv2 ucsf
#data = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/89d3ad98_May23-13-58-49_sd1000_dev_tune0/models/eval/training_124999/f5385048_Jun01-17-04-04_sd1000_INF_dev/predictions/pred.pt")

data = pd.DataFrame(data)
inf_embs = torch.stack(data["embeddings"].tolist())

In [ ]:
#db_data = db_data[db_data["label"].isin({0,1,4})]

In [ ]:
#db_embs = torch.stack(db_data["embeddings"].tolist())
#db_label = torch.tensor(db_data["label"].apply(lambda x: {0: 0, 1:1, 4:2}[x]).tolist())

db_label = torch.tensor(db_data["label"].apply(lambda x: {"tumor": 1, "normal":0}[x]))

In [ ]:
db_mean = db_embs.mean(dim=0)

inf_embs = inf_embs - db_mean
db_embs = db_embs - db_mean

inf_embs_norm = torch.nn.functional.normalize(inf_embs, dim=1)
db_embs_norm = torch.nn.functional.normalize(db_embs, dim=1)

In [ ]:
# KNN INFERENCE

knn_conf = OmegaConf.create({
    "testing":{"knn":{"knn_params":{
        "batch_size":128,
        "k": 200,
        "t": 0.07
}}}})

val_predictions = get_knn_logits(knn_conf, db_embs_norm, inf_embs_norm, db_label)
data["prediction"] = val_predictions[:,0]

In [ ]:
# GMM INFERENCE

#gmm = gmm_fit(db_embs_norm, n_components=32,
#        covariance_type='diag',
#        max_iter=200,
#        init_params='kmeans',
#        random_state=0)
#joblib.dump(gmm, "gmm_glioma2m_32.pkl")

gmm = joblib.load("silica_gmm/ours_ln/models/gmm_g2m_m32.pkl")

gmm_pred = gmm.predict(db_embs_norm)
cluster_pred = pd.DataFrame({"pred": gmm_pred, "label": db_data["label"]})
cluster_pred['label_bin'] = (cluster_pred['label'] == 'tumor').astype(int)
positive_rate = cluster_pred.groupby('pred')['label_bin'].mean()
mixture_score = np.array(positive_rate)

In [ ]:
mixture_score[4] = 0
mixture_score[-3] = 0

In [ ]:
gmm_score, gmm_responsibilities= gmm_inference(gmm, mixture_score, inf_embs_norm, mixture_hard=False, point_hard=False)
data["prediction"] = 1-gmm_score

In [ ]:
data["assignment"] = gmm_responsibilities.tolist()

In [ ]:
data[data["path"].str.contains("NIO_UCSF_285-8")]["assignment"].apply(lambda x: np.array(x).argmax()).value_counts()

In [ ]:
data[data["path"].str.contains("NIO_UCSF_285-8")]



In [ ]:
plt.hist(data[data["path"].str.contains("NIO_UCSF_285-8")]["assignment"].apply(lambda x: np.array(x).max()))


In [ ]:
#data["slide"] = data["path"].str.extract("(NIO_UM_[0-9]+[a-zA-Z]*-[0-9a-zA-Z]*)")
data["slide"] = data["path"].str.extract("(nio_mouse_[0-9]+[a-zA-Z]*-[0-9a-zA-Z]*)")
#data["slide"] = data["path"].str.extract("(NIO_[a-zA-Z]+_[0-9]+[a-zA-Z]*-[0-9a-zA-Z]*)")

In [ ]:
data[["path", "slide", "label","prediction"]].to_csv("silica_predictions/mouse_ours_g2m_cn_gmm32_softsoft_manual_override_pred.csv", index=False)
#data[["path", "slide", "label","prediction"]].to_csv("silica_predictions/ucsf_ours_g2m_cn_gmm32_softsoft_manual_override_pred.csv", index=False)


In [ ]:
import matplotlib

In [ ]:
def decode_cell_coord(in_str):
    in_patch = [int(i) for i in in_str.split("#")[-1].split("_")]
    patch_coord = [int(i) for i in in_str.split("#")[0].split("-")[-1].split("_")]
    return (patch_coord[0] + patch_coord[2] + in_patch[0]
            , patch_coord[1]*0.9 + patch_coord[3] + in_patch[1])

In [ ]:
def generate_viz(nion, slide, out_dir):
    nio950_pred = data[data["slide"]==f"{nion}-{slide}"]
    
    norm_pred = np.array(nio950_pred["prediction"])
    norm_pred = np.power(norm_pred, 1)
    
    plt.rcParams['font.size'] = 14  # Set default font size
    matplotlib.rcParams['pdf.fonttype'] = 42
    fig, ax = plt.subplots(1,1)
    ax.hist(1-norm_pred, color="#475569", zorder=2)
    ax.set_xlim(0,1)
    
    ax.xaxis.set_ticks_position('none')
    ax.yaxis.set_ticks_position('none')
    plt.ylabel("Cells")
    plt.xlabel("Tumor likelihood")
    ax.grid(axis='both',color="#cbd5e1", zorder=0)
    plt.tight_layout()
    plt.savefig(f"{out_dir}/{nion}-{slide}-hist.pdf")
    
    nio950_cells = np.array(nio950_pred["path"].apply(decode_cell_coord).tolist())
    
    #image = pydicom.dcmread("/nfs/turbo/umms-tocho/root_srh_db/UM/NIO_UM_950/3/mosaic/IMG00001.dcm").pixel_array
    
    if "UCSF" in nion:
        image_fn_cand = glob(f"/nfs/turbo/umms-tocho/root_srh_db/UCSF/{nion}/{slide}/mosaic*/img*_1*.dcm")
        assert len(image_fn_cand) > 0
        image = pydicom.dcmread(image_fn_cand[0]).pixel_array
    elif "UKK" in nion:
        image_fn_cand = glob(f"/nfs/turbo/umms-tocho/root_srh_db/UKK/{nion}/{slide}/mosaic*/IMG00001.dcm")
        assert len(image_fn_cand) == 1
        image = pydicom.dcmread(image_fn_cand[0]).pixel_array
    elif "mouse" in nion:
        image = pydicom.dcmread(f"/nfs/turbo/umms-tocho/data/db_nio_mouse/mouse/{nion}/{slide}/mosaics/IMG00001.dcm").pixel_array
    elif "UM" in nion:
        image = pydicom.dcmread(f"/nfs/turbo/umms-tocho/root_srh_db/UM/{nion}/{slide}/mosaic/IMG00001.dcm").pixel_array
    else:
        raise ValueError()

    strip_width=900
    strip_padding=50
    image = np.pad(image, ((strip_padding, strip_padding), (strip_padding, 0),(0,0)))[:,:-50,...]
    
    
    if image.dtype != np.uint8:
        image = np.clip(image, 0, 1)
        image = (image * 255).astype(np.uint8)
    if image.shape[-1] == 3:
        image_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    else:
        raise ValueError("Image must be RGB.")

    # Get colormap
    cmap = plt.get_cmap('RdYlGn')

    # Draw circles
    overlay = image_bgr.copy()
    for (y, x), p in zip(nio950_cells, norm_pred):
        color = np.array(cmap(p)[:3]) * 255  # RGBA -> RGB scaled
        color_bgr = tuple(map(int, color[::-1]))  # Convert to BGR and int
        cv2.circle(overlay, (int(x), int(y)), radius=10, color=color_bgr, thickness=-1)

    # Optional: blend if alpha is desired
    alpha = 0.5
    blended = cv2.addWeighted(overlay, alpha, image_bgr, 1 - alpha, 0)
    
    cv2.imwrite(f"{out_dir}/{nion}-{slide}.png", blended)
    fig, ax = plt.subplots(1,1,figsize=(12,12))
    ax.imshow(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))


In [ ]:
!ls silica_predictions/

In [ ]:
data = pd.read_csv("silica_predictions/ucsf_ours_g2m_cn_gmm32_softsoft_manual_override_pred.csv") # mouse_ours_g2m_cn200_knn_pred.csv") #

In [ ]:
generate_viz("NIO_UCSF_285", "8", out_dir=".")

In [ ]:
most_diff = pd.DataFrame({"slide": [
'NIO_UCSF_105297-10', 'NIO_UCSF_105297-9', 'NIO_UCSF_108811-1', 'NIO_UCSF_108811-5', 'NIO_UCSF_11330-6', 'NIO_UCSF_113587-2', 'NIO_UCSF_113587-4', 'NIO_UCSF_116351-4', 'NIO_UCSF_116351-5', 'NIO_UCSF_118425-1', 'NIO_UCSF_119323-1', 'NIO_UCSF_12123-1', 'NIO_UCSF_12123-5', 'NIO_UCSF_12157-2', 'NIO_UCSF_12158-2', 'NIO_UCSF_12164-1', 'NIO_UCSF_12175-2', 'NIO_UCSF_12187-5', 'NIO_UCSF_12209-1', 'NIO_UCSF_12209-5', 'NIO_UCSF_12223-1', 'NIO_UCSF_12225-1', 'NIO_UCSF_12225-5', 'NIO_UCSF_12228-1', 'NIO_UCSF_12243-1', 'NIO_UCSF_12244-13', 'NIO_UCSF_12259-1', 'NIO_UCSF_125932-8', 'NIO_UCSF_12629-4', 'NIO_UCSF_127633-10', 'NIO_UCSF_127633-7', 'NIO_UCSF_132140-9', 'NIO_UCSF_13394-1', 'NIO_UCSF_13394-2', 'NIO_UCSF_13394-3', 'NIO_UCSF_13394-4', 'NIO_UCSF_13394-6', 'NIO_UCSF_13394-7', 'NIO_UCSF_13394-8', 'NIO_UCSF_136601-2', 'NIO_UCSF_140320-1', 'NIO_UCSF_1453-1', 'NIO_UCSF_1453-2', 'NIO_UCSF_146634-6', 'NIO_UCSF_149609-11', 'NIO_UCSF_157889-2', 'NIO_UCSF_158976-11', 'NIO_UCSF_163842-8', 'NIO_UCSF_1641212-1', 'NIO_UCSF_171814-3', 'NIO_UCSF_178117-10', 'NIO_UCSF_184357-3', 'NIO_UCSF_188836-10', 'NIO_UCSF_189497-10', 'NIO_UCSF_189497-11', 'NIO_UCSF_190864-5', 'NIO_UCSF_197741-1', 'NIO_UCSF_197741-2', 'NIO_UCSF_204503-3', 'NIO_UCSF_215-1', 'NIO_UCSF_220-1', 'NIO_UCSF_232-7', 'NIO_UCSF_23403-4', 'NIO_UCSF_23403-5', 'NIO_UCSF_238-3', 'NIO_UCSF_240-3', 'NIO_UCSF_241-3', 'NIO_UCSF_252-3', 'NIO_UCSF_259-7', 'NIO_UCSF_264-1', 'NIO_UCSF_269-8', 'NIO_UCSF_281-1', 'NIO_UCSF_282-7', 'NIO_UCSF_2873-1', 'NIO_UCSF_2873-6', 'NIO_UCSF_288-15', 'NIO_UCSF_300-2', 'NIO_UCSF_302-2', 'NIO_UCSF_304-7', 'NIO_UCSF_313-9', 'NIO_UCSF_33134-1', 'NIO_UCSF_33134-2', 'NIO_UCSF_337-2', 'NIO_UCSF_340-6', 'NIO_UCSF_341-1', 'NIO_UCSF_343-10', 'NIO_UCSF_346-1', 'NIO_UCSF_352-1', 'NIO_UCSF_35214-4', 'NIO_UCSF_367-4', 'NIO_UCSF_3833-10', 'NIO_UCSF_3833-5', 'NIO_UCSF_386-2', 'NIO_UCSF_394-3', 'NIO_UCSF_413-1', 'NIO_UCSF_42790-5', 'NIO_UCSF_428-5', 'NIO_UCSF_44172-4', 'NIO_UCSF_468-10', 'NIO_UCSF_469-1', 'NIO_UCSF_471-5', 'NIO_UCSF_473-4', 'NIO_UCSF_48160-3', 'NIO_UCSF_485-8', 'NIO_UCSF_508-1', 'NIO_UCSF_50892-5', 'NIO_UCSF_516-5', 'NIO_UCSF_519-1', 'NIO_UCSF_519-5', 'NIO_UCSF_529-3', 'NIO_UCSF_533-2', 'NIO_UCSF_534-1', 'NIO_UCSF_53983-1', 'NIO_UCSF_543-6', 'NIO_UCSF_5454-1', 'NIO_UCSF_5454-10', 'NIO_UCSF_5454-11', 'NIO_UCSF_5454-2', 'NIO_UCSF_5454-3', 'NIO_UCSF_5454-4', 'NIO_UCSF_5454-5', 'NIO_UCSF_5454-6', 'NIO_UCSF_5454-7', 'NIO_UCSF_5454-8', 'NIO_UCSF_5454-9', 'NIO_UCSF_554-2', 'NIO_UCSF_555-2', 'NIO_UCSF_559-2', 'NIO_UCSF_561-6', 'NIO_UCSF_563-12', 'NIO_UCSF_563-8', 'NIO_UCSF_575-3', 'NIO_UCSF_577-2', 'NIO_UCSF_577-6', 'NIO_UCSF_60473-6', 'NIO_UCSF_614-1', 'NIO_UCSF_614-2', 'NIO_UCSF_614-3', 'NIO_UCSF_6550-4', 'NIO_UCSF_6550-8', 'NIO_UCSF_66486-8', 'NIO_UCSF_708-1', 'NIO_UCSF_8662-3', 'NIO_UKK_1025-1', 'NIO_UKK_15-2', 'NIO_UKK_155-1', 'NIO_UKK_187-1', 'NIO_UKK_187-3', 'NIO_UKK_200-1', 'NIO_UKK_207-1', 'NIO_UKK_247-1', 'NIO_UKK_259-1', 'NIO_UKK_261-1', 'NIO_UKK_265-1', 'NIO_UKK_267-1', 'NIO_UKK_267-2', 'NIO_UKK_361-2', 'NIO_UKK_373-1', 'NIO_UKK_437-1', 'NIO_UKK_497-1', 'NIO_UKK_503-1', 'NIO_UKK_509-1', 'NIO_UKK_514-1', 'NIO_UKK_525-1', 'NIO_UKK_529-1', 'NIO_UKK_531-1', 'NIO_UKK_554-1', 'NIO_UKK_577-1', 'NIO_UKK_581-1', 'NIO_UKK_592-1', 'NIO_UKK_602-1', 'NIO_UKK_616-1', 'NIO_UKK_641-1', 'NIO_UKK_684-1', 'NIO_UKK_693-1', 'NIO_UKK_72-12', 'NIO_UKK_72-2', 'NIO_UKK_730-1', 'NIO_UKK_74-1', 'NIO_UKK_762-1', 'NIO_UKK_790-1', 'NIO_UKK_791-1', 'NIO_UKK_800-1', 'NIO_UKK_830-1', 'NIO_UKK_906-2'
]})
trouble = pd.DataFrame({"slide": [
    'NIO_UCSF_117375-10',  'NIO_UCSF_117375-9',  'NIO_UCSF_124344-2',  'NIO_UCSF_124344-6',  'NIO_UCSF_125932-8',  'NIO_UCSF_127633-10',  'NIO_UCSF_127633-5',  'NIO_UCSF_129158-5',  'NIO_UCSF_132140-7',  'NIO_UCSF_132140-8',  'NIO_UCSF_132140-9',  'NIO_UCSF_133469-9',  'NIO_UCSF_14635-13',  'NIO_UCSF_146634-7',  'NIO_UCSF_147101-9',  'NIO_UCSF_152570-4',  'NIO_UCSF_152570-6',  'NIO_UCSF_168818-5',  'NIO_UCSF_168818-6',  'NIO_UCSF_168818-8',  'NIO_UCSF_170345-3',  'NIO_UCSF_171814-5',  'NIO_UCSF_171814-7',  'NIO_UCSF_173244-11',  'NIO_UCSF_176812-4',  'NIO_UCSF_1771257-4',  'NIO_UCSF_179469-14',  'NIO_UCSF_179469-2',  'NIO_UCSF_179469-3',  'NIO_UCSF_179469-4',  'NIO_UCSF_18179-1',  'NIO_UCSF_18179-2',  'NIO_UCSF_18179-5',  'NIO_UCSF_18179-6',  'NIO_UCSF_18179-7',  'NIO_UCSF_18179-8',  'NIO_UCSF_189497-11',  'NIO_UCSF_189497-8',  'NIO_UCSF_190864-6',  'NIO_UCSF_19383-7',  'NIO_UCSF_194193-5',  'NIO_UCSF_198380-6',  'NIO_UCSF_198380-7',  'NIO_UCSF_203239-4',  'NIO_UCSF_205943-3',  'NIO_UCSF_214942-1',  'NIO_UCSF_214942-2',  'NIO_UCSF_215-6',  'NIO_UCSF_219-6',  'NIO_UCSF_221-2',  'NIO_UCSF_224-6',  'NIO_UCSF_233-3',  'NIO_UCSF_234-4',  'NIO_UCSF_23403-10',  'NIO_UCSF_254-7',  'NIO_UCSF_260-6',  'NIO_UCSF_261-3',  'NIO_UCSF_261-4',  'NIO_UCSF_26162-3',  'NIO_UCSF_266-3',  'NIO_UCSF_266-5',  'NIO_UCSF_266-6',  'NIO_UCSF_268-2',  'NIO_UCSF_277-5',  'NIO_UCSF_285-8',  'NIO_UCSF_291-4',  'NIO_UCSF_299-2',  'NIO_UCSF_302-5',  'NIO_UCSF_306-6',  'NIO_UCSF_309-5',  'NIO_UCSF_313-9',  'NIO_UCSF_314-4',  'NIO_UCSF_314-5',  'NIO_UCSF_315-1',  'NIO_UCSF_322-1',  'NIO_UCSF_323-4',  'NIO_UCSF_331-5',  'NIO_UCSF_341-5',  'NIO_UCSF_349-5',  'NIO_UCSF_349-7',  'NIO_UCSF_356-6',  'NIO_UCSF_357-2',  'NIO_UCSF_371-1',  'NIO_UCSF_390-3',  'NIO_UCSF_395-4',  'NIO_UCSF_403-2',  'NIO_UCSF_405-11',  'NIO_UCSF_405-4',  'NIO_UCSF_405-6',  'NIO_UCSF_406-4',  'NIO_UCSF_427-5',  'NIO_UCSF_434-10',  'NIO_UCSF_434-2',  'NIO_UCSF_434-3',  'NIO_UCSF_442-2',  'NIO_UCSF_444-4',  'NIO_UCSF_465-4',  'NIO_UCSF_467-5',  'NIO_UCSF_467-7',  'NIO_UCSF_468-12',  'NIO_UCSF_486-3',  'NIO_UCSF_49407-8',  'NIO_UCSF_501-8',  'NIO_UCSF_513-6',  'NIO_UCSF_530-3',  'NIO_UCSF_530-4',  'NIO_UCSF_534-9',  'NIO_UCSF_536-5',  'NIO_UCSF_539-6',  'NIO_UCSF_543-3',  'NIO_UCSF_543-6',  'NIO_UCSF_547-7',  'NIO_UCSF_556-5',  'NIO_UCSF_557-3',  'NIO_UCSF_561-6',  'NIO_UCSF_574-3',  'NIO_UCSF_574-4',  'NIO_UCSF_69567-2',  'NIO_UCSF_83304-4',  'NIO_UCSF_83304-5',  'NIO_UCSF_91565-11',  'NIO_UCSF_941-1',  'NIO_UCSF_941-2',  'NIO_UCSF_96371-2'
]})

good_viz = pd.DataFrame({"slide": [
    "NIO_UCSF_179469-6", "NIO_UCSF_176812-12", "NIO_UCSF_49407-7", "NIO_UCSF_74388-5", "NIO_UCSF_152570-5", "NIO_UCSF_180426-3", "NIO_UCSF_7399-1", "NIO_UCSF_114346-3", "NIO_UCSF_114346-4", "NIO_UCSF_121503-7", "NIO_UCSF_14147-1", "NIO_UCSF_7777-6", "NIO_UCSF_116351-5", "NIO_UCSF_14635-1", "NIO_UCSF_88550-6", "NIO_UCSF_165353-2", "NIO_UCSF_171814-1", "NIO_UCSF_192968-7", "NIO_UCSF_1231014-2", "NIO_UCSF_4392-9", "NIO_UCSF_6550-3", "NIO_UCSF_173244-2", "NIO_UCSF_6550-2", "NIO_UCSF_25642-4", "NIO_UCSF_30_2781-2", "NIO_UCSF_4723-4", "NIO_UCSF_44172-2", "NIO_UCSF_108811-2", "NIO_UCSF_165353-1", "NIO_UCSF_167596-9", "NIO_UCSF_63756-1", "NIO_UCSF_147101-7", "NIO_UCSF_151922-5", "NIO_UCSF_197741-1", "NIO_UCSF_84278-3", "NIO_UCSF_129158-6", "NIO_UCSF_149609-9", "NIO_UCSF_166498-4", "NIO_UCSF_178117-1", "NIO_UCSF_65823-1", "NIO_UCSF_14859-1", "NIO_UCSF_16112-2", "NIO_UCSF_91565-9", "NIO_UCSF_162688-3", "NIO_UCSF_78641-3", "NIO_UCSF_108811-4"]})


In [ ]:
!mkdir -p silica_predictions/random/g2m_gmm_ss



In [ ]:
rand_slide = data["slide"].drop_duplicates().sample(200).sort_values()
slides = rand_slide.drop_duplicates().str.split("-")
for s in tqdm(slides):
    try:
        generate_viz(*s, out_dir="silica_predictions/random/g2m_gmm_ss")
    except:
        print(s)
        
        

In [ ]:
!mkdir -p silica_predictions/trouble/g2m_knn200
!mkdir -p  silica_predictions/good_viz/g2m_knn200
!mkdir  -p silica_predictions/diff/g2m_knn200

In [ ]:
slides = trouble["slide"].drop_duplicates().str.split("-")
for s in tqdm(slides):
    try:
        generate_viz(*s, out_dir="silica_predictions/trouble/g2m_knn200")
    except:
        print(s)

slides = good_viz["slide"].drop_duplicates().str.split("-")
for s in tqdm(slides):
    try:
        generate_viz(*s, out_dir="silica_predictions/good_viz/g2m_knn200")
    except:
        print(s)

slides = most_diff["slide"].drop_duplicates().str.split("-")
for s in tqdm(slides):
    try:
        generate_viz(*s, out_dir="silica_predictions/diff/g2m_knn200")
    except:
        print(s)

In [ ]:
!mkdir -p silica_predictions/mouse/g2m_knn200
#!mkdir -p silica_predictions/good_viz/g2m_gmm32_ss
mouse_viz = data["slide"].drop_duplicates()
slides = mouse_viz.str.split("-")
for s in tqdm(slides):
    #try:
        generate_viz(*s, out_dir="silica_predictions/mouse/g2m_knn200")
    #except:
    #    print(s)

In [ ]:
data

In [ ]:
generate_viz("NIO_UCSF_91565","9", out_dir="./delete_me")

In [ ]:
def mean_class_accuracy(cm: np.ndarray) -> float:
    with np.errstate(divide='ignore', invalid='ignore'):
        class_accuracy = np.diag(cm) / cm.sum(axis=1)
        class_accuracy = np.nan_to_num(class_accuracy)  # set NaNs from zero division to 0
    return class_accuracy.mean()

In [ ]:
 cm = np.array([[36+4+1+1+1+2+19+2+1+43+1+2+4+17+2+2+1+43+5, 2],[2,25]])

In [ ]:
mean_class_accuracy(cm)

In [ ]:
cm

In [ ]:
mean_class_accuracy(np.array(([cm[:-1,:-1].sum(), cm[:-1,-1].sum()], [cm[-1,:-1].sum(), cm[-1,-1]])))